# 第7章 数据输入输出：同一份数据，不同的存法

**目标**：观察 CSV、Excel、Parquet 的结构；掌握各自的读取参数；比较读取耗时；理解列式存储、文件内部行组与目录分区。

先制作一套 20,000 行虚拟行情，观察其前 6 行，再用同一套数据比较格式，最后回到教材的真实行情。虚拟证券、价格和连续分钟时间仅供教学，不代表交易日历或真实行情。

教学数据与结果文件直接放在本章 `data/`，不另设输出目录；重复运行会覆盖同名教学文件，原始 `eod_data.csv` 只读。所需库为 numpy、pandas、openpyxl、pyarrow、Jupyter，请选择课程根目录 `.venv` 内核。

## 0. 路径与环境
支持从课程根目录、本章目录或本章 notebooks 目录运行。

In [1]:
from pathlib import Path
import sys
import platform
import numpy as np
import pandas as pd
import pyarrow
import openpyxl
from IPython.display import display

cwd = Path.cwd()
if cwd.name == "notebooks":
    CHAPTER = cwd.parent
elif cwd.name == "Chapter7 - 数据输入输出":
    CHAPTER = cwd
else:
    CHAPTER = cwd / "Chapter7 - 数据输入输出"
assert (CHAPTER / "data/eod_data.csv").is_file(), "请从课程根目录或本章目录运行"
DATA = CHAPTER / "data"
print("Python:", sys.version.split()[0], "|", platform.platform())
print("解释器:", sys.executable)
print("pandas:", pd.__version__, "pyarrow:", pyarrow.__version__,
      "openpyxl:", openpyxl.__version__)

Python: 3.11.15 | Windows-10-10.0.26200-SP0
解释器: D:\F Python data analysis\python-for-finance\.venv\Scripts\python.exe
pandas: 2.3.3 pyarrow: 25.0.1 openpyxl: 3.1.5


## 1. 一份虚拟行情，写成三种格式
`scripts/prepare_data.py` 固定随机种子 7，将同一个 DataFrame 保存为 `data/quotes.csv`、`data/quotes.xlsx`、`data/quotes.parquet`。
每行是一条分钟观测：日期、6 位代码、市场、价格、成交量、买价、卖价、评分。评分每隔 17 行缺失一次。
前导零、中文、日期和缺失值用于观察类型信息。小样本直接 `head(6)`，不另建目录。

额外生成 `data/quotes_by_market/`，按市场分成两个目录，用于解释分区；它不参与三种单文件的速度比较。生成和写入不包含在计时内。

In [2]:
sys.path.insert(0, str(CHAPTER / "scripts"))
from prepare_data import prepare, FORMATS

N_ROWS = 20_000
quotes, DATA = prepare(N_ROWS)
display(quotes.head(6))
display(quotes.dtypes.to_frame("原始类型"))

,date,symbol,market,price,volume,bid,ask,score
0,2025-01-01 00:00:00,000001,模拟市场甲,95.04,8049,95.03,95.05,NaN
1,2025-01-01 00:01:00,000002,模拟市场甲,66.25,5299,66.24,66.26,0.486408
2,2025-01-01 00:02:00,000003,模拟市场乙,71.57,7942,71.56,71.58,-0.203099
3,2025-01-01 00:03:00,000001,模拟市场甲,90.74,9677,90.73,90.75,-0.537259
4,2025-01-01 00:04:00,000002,模拟市场甲,62.04,489,62.03,62.05,0.662349
5,2025-01-01 00:05:00,000003,模拟市场乙,79.81,8118,79.80,79.82,0.460094


,原始类型
date,datetime64[ns]
symbol,object
market,object
price,float64
volume,int64
bid,float64
ask,float64
score,float64


### 文件预览：三种文件装着同一张表
下图由本章真实生成文件绘制，是教学预览界面，不是第三方软件截图。Parquet 通过读取器还原成表格，不能用记事本直接看表。

![同一份数据的三种预览](../latex/figures/formats_preview.png)

三种预览展示相同的前 3 行和全部 8 列，包含 `market`、`bid` 和 `ask`，列顺序完全一致。

## 2. CSV：文本、类型与分块
CSV 本身不保存 pandas 类型；证券代码可能被推断为整数，日期可能仍是文本。
`dtype` 保留代码前导零，`parse_dates` 解析日期，`na_values` 约定缺失标记。
文件中含分隔符或换行的字段需要引号规则，因此不要自行 `split(",")` 代替 CSV 解析器。

先用 `open` 查看同一份行情 CSV 的表头与前 6 行。`with` 自动关闭文件；`r` 读取（默认）、`w` 覆盖、`a` 追加。中文文本明确使用 UTF-8。

In [3]:
with open(DATA / "quotes.csv", encoding="utf-8") as f:
    print("".join(f.readline() for _ in range(7)))  # 表头 + 前 6 行
naive = pd.read_csv(DATA / "quotes.csv", nrows=6)
csv_back = pd.read_csv(DATA / "quotes.csv", nrows=6, encoding="utf-8",
                       dtype={"symbol": str}, parse_dates=["date"])
display(pd.DataFrame({"自动推断代码": naive["symbol"],
                      "明确字符串代码": csv_back["symbol"]}))

date,symbol,market,price,volume,bid,ask,score
2025-01-01 00:00:00,000001,模拟市场甲,95.04,8049,95.03,95.05,
2025-01-01 00:01:00,000002,模拟市场甲,66.25,5299,66.24,66.26,0.486408
2025-01-01 00:02:00,000003,模拟市场乙,71.57,7942,71.56,71.58,-0.203099
2025-01-01 00:03:00,000001,模拟市场甲,90.74,9677,90.73,90.75,-0.537259
2025-01-01 00:04:00,000002,模拟市场甲,62.04,489,62.03,62.05,0.662349
2025-01-01 00:05:00,000003,模拟市场乙,79.81,8118,79.8,79.82,0.460094



,自动推断代码,明确字符串代码
0,1,000001
1,2,000002
2,3,000003
3,1,000001
4,2,000002
5,3,000003


### 2.1 只取列和分块汇总
`usecols` 减少构造的列与内存，但 CSV 仍需扫描文本；`chunksize` 每次返回一块，适合累加、计数等可分解的任务。
把所有块再拼起来仍会占用整表内存。这里核对成交量总和。

In [4]:
volume_total = 0
for chunk in pd.read_csv(DATA / "quotes.csv", usecols=["volume"], chunksize=5000):
    volume_total += chunk["volume"].sum()
assert volume_total == quotes["volume"].sum()
print("分块累计成交量:", volume_total)

分块累计成交量: 101447171


## 3. Excel：工作簿与工作表
适合人工查看和交付，多工作表必须共用一个 `ExcelWriter`。对同一路径连续调用两次 `to_excel` 默认会重建文件，不能当成追加工作表。
`sheet_name` 选表、`usecols` 选列、`dtype` 保护代码。pandas 读取的是表格数据，不是完整保留样式、图表和公式计算行为的工作簿编辑器。

In [5]:
summary = quotes.groupby("symbol", as_index=False)["volume"].sum()
with pd.ExcelWriter(DATA / "report.xlsx", engine="openpyxl") as writer:
    quotes.head(6).to_excel(writer, sheet_name="行情", index=False)
    summary.to_excel(writer, sheet_name="汇总", index=False)
with pd.ExcelFile(DATA / "report.xlsx", engine="openpyxl") as book:
    print("工作表:", book.sheet_names)
    display(pd.read_excel(book, sheet_name="汇总", dtype={"symbol": str}))

工作表: ['行情', '汇总']


,symbol,volume
0,000001,33791791
1,000002,33684785
2,000003,33970595


## 4. Parquet：列式二进制存储
**示意**：行式把 `(代码1, 价格1, 成交量1)` 放在一起；列式把一批价格放在一起、一批成交量放在一起。
Parquet 实际按 **row group（行组）→ column chunk（列块）→ page（页）** 组织，不是整个文件每列只有一个连续块。

- 同列类型一致、值可能重复，有利于编码和压缩；文件包含模式与类型信息。
- `columns` 在读取时选择列，减少无关列的读取和解码。先全表读入再切列做不到这一点。
- `filters` 由 pyarrow 执行筛选；元数据统计允许时可以跳过行组，但不保证每个条件都能减少磁盘读取。
- Parquet 不是 Excel 工作簿，也不是可逐行更新的数据库。本例修改后重写文件；大型数据集可增加分区文件。

以下与生成脚本一致，使用 pyarrow、Snappy 压缩和每组 5,000 行。

In [6]:
parquet_back = pd.read_parquet(DATA / "quotes.parquet", engine="pyarrow")
display(parquet_back.head(6))
display(parquet_back.dtypes.to_frame("读回类型"))
import pyarrow.parquet as pq
metadata = pq.ParquetFile(DATA / "quotes.parquet").metadata
print("行数:", metadata.num_rows, "列数:", metadata.num_columns,
      "行组数:", metadata.num_row_groups)

,date,symbol,market,price,volume,bid,ask,score
0,2025-01-01 00:00:00,000001,模拟市场甲,95.04,8049,95.03,95.05,NaN
1,2025-01-01 00:01:00,000002,模拟市场甲,66.25,5299,66.24,66.26,0.486408
2,2025-01-01 00:02:00,000003,模拟市场乙,71.57,7942,71.56,71.58,-0.203099
3,2025-01-01 00:03:00,000001,模拟市场甲,90.74,9677,90.73,90.75,-0.537259
4,2025-01-01 00:04:00,000002,模拟市场甲,62.04,489,62.03,62.05,0.662349
5,2025-01-01 00:05:00,000003,模拟市场乙,79.81,8118,79.80,79.82,0.460094


,读回类型
date,datetime64[ns]
symbol,object
market,object
price,float64
volume,int64
bid,float64
ask,float64
score,float64


行数: 20000 列数: 8 行组数: 4


In [7]:
selected = pd.read_parquet(DATA / "quotes.parquet", engine="pyarrow",
                           columns=["symbol", "price"],
                           filters=[("symbol", "==", "000001")])
expected = quotes.loc[quotes["symbol"] == "000001", ["symbol", "price"]].reset_index(drop=True)
pd.testing.assert_frame_equal(selected, expected)
display(selected.head(3))

,symbol,price
0,000001,95.04
1,000001,90.74
2,000001,85.02


### 4.1 单个文件内部：行组、列块、元数据
下图的行数、列数、行组大小和压缩方式来自 `quotes.parquet` 的实际元数据。方块只表示结构，不按字节比例绘制。

![Parquet 文件内部结构](../latex/figures/parquet_layout.png)

### 4.2 多个文件组成一个数据集：按市场分区
`quotes.parquet` 是一个文件；`quotes_by_market/` 是包含多个文件的目录。目录中的 `market=值` 是 Hive 风格分区约定，不是每个 Parquet 文件都必须这样摆放。
本例文件内不重复存储 `market` 列，读取数据集根目录时从目录名恢复。图中的目录名、文件大小与行数均来自实际文件。

![Parquet 分区目录](../latex/figures/parquet_directory.png)

In [8]:
partition_root = DATA / "quotes_by_market"
for path in sorted(partition_root.rglob("*.parquet")):
    print(path.relative_to(DATA), "行数:", pq.ParquetFile(path).metadata.num_rows)

partitioned = pd.read_parquet(partition_root, engine="pyarrow")
# 多文件数据集不保证原始行顺序；本例 date 唯一，因此排序后核对。
partitioned = partitioned.sort_values("date").reset_index(drop=True)
partitioned = partitioned.loc[:, quotes.columns]
partitioned["market"] = partitioned["market"].astype(quotes["market"].dtype)
pd.testing.assert_frame_equal(partitioned, quotes)

market_a = pd.read_parquet(partition_root, engine="pyarrow",
                          filters=[("market", "==", "模拟市场甲")])
assert len(market_a) == (quotes["market"] == "模拟市场甲").sum()
display(market_a.head(3))

quotes_by_market\market=模拟市场乙\part-0.parquet 行数: 6666
quotes_by_market\market=模拟市场甲\part-0.parquet 行数: 13334


,date,symbol,price,volume,bid,ask,score,market
0,2025-01-01 00:00:00,000001,95.04,8049,95.03,95.05,NaN,模拟市场甲
1,2025-01-01 00:01:00,000002,66.25,5299,66.24,66.26,0.486408,模拟市场甲
2,2025-01-01 00:03:00,000001,90.74,9677,90.73,90.75,-0.537259,模拟市场甲


按市场筛选时，引擎可以根据目录名排除另一个市场的文件；这叫分区裁剪。它和文件内部利用统计信息跳过行组是两件事。
分区列可能被读取为分类类型，读取顺序也可能改变，所以核对时显式处理类型和顺序。
这里只示范目录组织，不把分区数据集的读取时间混进单文件的速度排名。

## 5. 实验前先验证：比较的是同一份数据吗？
读取器负责恢复代码与日期，计时包含这些读取参数的代价。
比较之前统一列顺序、日期精度及字符串表示；统一操作和断言放在计时之外。
检查行列、类型、缺失位置与值；浮点数采用 `rtol=1e-9, atol=1e-8`，避免把文本往返的微小误差当成数据损坏。
这不意味着可以忽略所有类型差异：代码前导零丢失仍会验证失败。

In [9]:
def read_csv():
    return pd.read_csv(DATA / "quotes.csv", encoding="utf-8",
                       dtype={"symbol": str}, parse_dates=["date"])

def read_excel():
    return pd.read_excel(DATA / "quotes.xlsx", sheet_name="quotes", engine="openpyxl",
                         dtype={"symbol": str}, parse_dates=["date"])

def read_parquet():
    return pd.read_parquet(DATA / "quotes.parquet", engine="pyarrow")

readers = {"CSV": read_csv, "Excel": read_excel, "Parquet": read_parquet}

def canonical(frame):
    result = frame.loc[:, quotes.columns].copy()
    result["date"] = pd.to_datetime(result["date"]).astype("datetime64[ns]")
    result[["symbol", "market"]] = result[["symbol", "market"]].astype("string")
    return result

reference = canonical(quotes)
for name, reader in readers.items():
    restored = canonical(reader())
    pd.testing.assert_frame_equal(restored, reference, check_exact=False, rtol=1e-9, atol=1e-8)
    print(name, "往返验证通过", restored.shape)

CSV 往返验证通过 (20000, 8)


Excel 往返验证通过 (20000, 8)
Parquet 往返验证通过 (20000, 8)


## 6. 同样的全表，读取速度一样吗？
每种格式先预读一次（上节已完成），再进行 5 轮随机顺序读取。记录每次耗时，报告中位数、最小值、最大值和文件体积。
计时只包围“读取文件得到 DataFrame”，不包含生成、写入、断言、显示或释放结果。单位为毫秒，体积为 KiB（1024 字节）。

**这是有缓存条件下的端到端读取实验**：未清空操作系统缓存，不是冷启动磁盘带宽测试，也没有测量峰值内存。
CSV 不压缩；Excel 使用 xlsx 容器自身的压缩；Parquet 使用 Snappy。因此比较的是这些常用配置，不能单独归因为存储布局。

In [10]:
from time import perf_counter

REPEATS = 5

def benchmark(tasks):
    order_rng = np.random.default_rng(2026)
    records = []
    for repeat in range(REPEATS):
        for name in order_rng.permutation(list(tasks)):
            start = perf_counter()
            result = tasks[name]()
            elapsed_ms = (perf_counter() - start) * 1000
            records.append({"format": name, "repeat": repeat + 1, "read_ms": elapsed_ms})
            del result
    return pd.DataFrame(records)

timings = benchmark(readers)
result = timings.groupby("format")["read_ms"].agg(["median", "min", "max"])
result["size_KiB"] = [(DATA / f"quotes.{FORMATS[name]}").stat().st_size / 1024
                       for name in result.index]
result = result.sort_values("median")
display(result.round(2))
timings.to_csv(DATA / "read_timings.csv", index=False)
result.to_csv(DATA / "read_summary.csv")
print(f"本次 {N_ROWS:,} 行 × {quotes.shape[1]} 列；最快读取中位数：{result.index[0]}。")
print("请根据自己机器上的表格回答，不要把这次排名推广到所有数据。")

,median,min,max,size_KiB
format,,,,
Parquet,3.07,2.83,3.30,744.37
CSV,13.61,13.32,13.83,1474.90
Excel,809.80,783.10,833.93,907.12


本次 20,000 行 × 8 列；最快读取中位数：Parquet。
请根据自己机器上的表格回答，不要把这次排名推广到所有数据。


### 6.1 只需要价格和成交量：读完再切，还是读取时选列？
CSV 的 `usecols` 与 Parquet 的 `columns` 对应相同两列；另设 Parquet 全表后切列作为对照。
先核对返回内容完全一致，再预热、计时。只比较同一任务内的耗时；这里少做了日期和代码解析，不能把它与全表实验当成只改变一个因素。
这张表刻意保留精确值，便于课堂记录和复算；不预设列裁剪一定快多少。

In [11]:
wanted = ["price", "volume"]
subset_readers = {
    "CSV usecols": lambda: pd.read_csv(DATA / "quotes.csv", usecols=wanted),
    "Parquet columns": lambda: pd.read_parquet(DATA / "quotes.parquet", engine="pyarrow", columns=wanted),
    "Parquet full then slice": lambda: read_parquet()[wanted],
}
for reader in subset_readers.values():
    pd.testing.assert_frame_equal(reader(), quotes[wanted], check_exact=False, rtol=1e-9, atol=1e-8)
subset_timings = benchmark(subset_readers)
display(subset_timings.groupby("format")["read_ms"].agg(["median", "min", "max"]).round(2))
subset_timings.to_csv(DATA / "column_timings.csv", index=False)

,median,min,max
format,,,
CSV usecols,8.50,8.13,8.63
Parquet columns,1.30,1.09,1.57
Parquet full then slice,2.53,2.34,2.86


### 6.2 如何解释差异？
1. 分别查看最快、最小文件和耗时波动最大的格式，它们是否相同？
2. Excel 要解释工作簿结构，文本要解析数字和日期，Parquet 要读取元数据并解压列块；这些工作量不同。
3. 小文件可能由初始化开销主导；宽表只取少数列时，列式存储通常更有发挥空间。
4. 改变行数、列数、压缩、缺失比例、字符串重复率或引擎，结果都可能改变。不能根据“二进制”三个字推断总是更小更快。

| 场景 | 可选格式 | 操作重点 |
|---|---|---|
| 通用平面表交换 | CSV | 编码、分隔符、类型、分块 |
| 人工整理和报告 | Excel | 工作表、ExcelWriter、引擎 |
| 反复分析大表和选列 | Parquet | 类型、压缩、columns、filters |

## 7. 回到真实数据：读入 → 计算 → 保存 → 验证
保留原教材行情案例。日期索引在输出时显式变成 `date` 列，使文件的数据约定清楚。
收益率和滚动波动率沿用前几章定义，Parquet 保存中间表，Excel 交付给人阅读。

In [12]:
raw = pd.read_csv(DATA / "eod_data.csv", index_col=0, parse_dates=True).dropna(subset=["SPY"])
rets = np.log(raw / raw.shift(1))
vol = (rets["SPY"].rolling(21).std() * np.sqrt(252)).dropna()
vol_table = vol.rename("vol21").rename_axis("date").reset_index()
vol_table.to_parquet(DATA / "spy_vol21.parquet", index=False, engine="pyarrow")
vol_table.to_excel(DATA / "spy_vol21.xlsx", index=False, engine="openpyxl")
back = pd.read_parquet(DATA / "spy_vol21.parquet", engine="pyarrow")
pd.testing.assert_frame_equal(back, vol_table)
display(back.tail(3))

,date,vol21
2114,2018-06-27,0.099753
2115,2018-06-28,0.090127
2116,2018-06-29,0.087729


## 8. 课堂练习与课后任务
1. 用记事本打开 CSV，用 Excel 打开 xlsx，用 pandas 预览 Parquet；找到相同的第一条记录和一个缺失值。Parquet 为什么不适合记事本查看？
2. 去掉 CSV 的 `dtype` 参数，观察 `000001`。再对比日期列的类型，说明“看起来一样”和“类型一致”的差别。
3. 把成交量汇总和前 6 行行情保存到同一 Excel 工作簿，重新打开并列出工作表名称。
4. 分别用 2,000 和 20,000 行重做全部三种格式实验，记录版本、行列数、文件体积、中位数及极值。每轮先另存结果，以免被覆盖。
5. 选做：在生成器中增加 20 个数值列，比较 Parquet 全表读取和只读两列。说明改变的是哪些变量。
6. 选做：将第6章计算得到的 beta 表写为 CSV 和 Parquet，分别读回验证；说明索引如何保存。

<details><summary>检查提示</summary>

- 文本类型提示：`pd.read_csv(path, dtype={"symbol": str}, parse_dates=["date"])`。
- 两张表在同一个 `with pd.ExcelWriter(...) as writer:` 中分别调用 `to_excel(writer, sheet_name=..., index=False)`。
- 数值往返用 `pd.testing.assert_frame_equal(..., check_exact=False, rtol=1e-9, atol=1e-8)`，但不能跳过代码、日期和缺失值检查。
- 时间越短越好与文件越小越好是两个问题；格式选择还取决于接收者和读取任务。

</details>

## 参考资料
教材第9章为原始课程基础；新增格式说明参照官方文档（2026-09-24 查阅）：
- [pandas read_csv](https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html)
- [pandas ExcelWriter](https://pandas.pydata.org/docs/reference/api/pandas.ExcelWriter.html)
- [pandas read_parquet](https://pandas.pydata.org/docs/reference/api/pandas.read_parquet.html)
- [Apache Parquet 概述](https://parquet.apache.org/docs/overview/)
- [Apache Parquet 文件结构](https://parquet.apache.org/docs/file-format/)